# Data quality and Assessment

First lets assess the data

In [7]:
import pandas as pd
from sklearn.impute import KNNImputer,SimpleImputer

In [8]:
df = pd.read_csv('dataset_clean_tidy.csv')
df.info(verbose=True)
print("Completeness: ", round((1 - df.isna().sum().sum() / df.size) * 100, 2), "%")

<class 'pandas.DataFrame'>
RangeIndex: 9105 entries, 0 to 9104
Data columns (total 24 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       8650 non-null   float64
 1   sex       9105 non-null   str    
 2   num_co    9105 non-null   int64  
 3   diabetes  9105 non-null   int64  
 4   dementia  9105 non-null   int64  
 5   meanbp    9104 non-null   float64
 6   hrt       9104 non-null   float64
 7   resp      9104 non-null   float64
 8   temp      9104 non-null   float64
 9   crea      9038 non-null   float64
 10  sod       9104 non-null   float64
 11  adlsc     9105 non-null   float64
 12  hday      9105 non-null   int64  
 13  ca        9105 non-null   str    
 14  scoma     9104 non-null   float64
 15  dzgroup   9105 non-null   str    
 16  aps       9104 non-null   float64
 17  wblc      8883 non-null   float64
 18  pafi      5919 non-null   float64
 19  income    5008 non-null   str    
 20  race      9063 non-null   str    
 21  ad

### Missing Data Analysis and Imputation Strategy

A preliminary Data Quality assessment reveals an overall dataset completeness of 94.83%, which generally indicates a highly robust and well-maintained clinical dataset. However, overall completeness can be a misleading metric if missingness is not uniformly distributed.

Indeed, a deeper column-wise analysis shows that the data lack is highly localized. The majority of missing entries are concentrated in `pafi`, `income`, and `adls`, with minor gaps in `age` and `wblc` (around 5% of the 9,105 total records). Since our primary objective is to train a model capable of predicting in-hospital mortality based strictly on day-3 clinical data, we must carefully evaluate how to handle these specific variables.

**1. Variables Excluded from the Model (`adls`)**
* **`adls` (High Missingness & Subjectivity Risk):** As noted in the dataset documentation, the Activities of Daily Living score reported by a surrogate (`adls`) is missing in approximately 50% of the records. Furthermore, because this metric is not a direct, objective clinical measurement taken by medical staff, but rather a subjective assessment provided by a third party, attempting to impute such a massive gap would likely introduce significant noise and pollute our dataset. Therefore, we have decided to exclude this column entirely from the analysis.

**2. Imputing `pafi` via Median (Expert Guidance hints.md)**
`pafi` (PaO₂/FiO₂ ratio) is a critical oxygenation index; a lower score indicates worse lung function. Following explicit guidance from clinical informatics experts, we note that the distribution of this marker is strictly right-skewed, featuring extreme high values in stable patients, while typical values for moderately ill hospitalized patients hover around 333 mmHg. Because of this pronounced asymmetry, we rely on **Median Imputation**. This mathematically safe approach bridges the data gaps without allowing extreme respiratory anomalies (the long right tail) to skew the imputed values.

**3. Imputing `income` via Mode (Most Frequent, hints.md)**
`income` represents a self-reported annual income category (a nominal variable). Although it suffers from a high missingness rate, standard demographic data practices as recommended by domain experts dictate that missing income brackets should be assigned the most common category observed within the study population. We therefore apply a `SimpleImputer` using the `most_frequent` strategy, retaining this socio-economic indicator without fabricating artificial distributions.

**4. Imputation via `KNNImputer` for Remaining Clinical Metrics**
To properly fill the missing gaps in our continuous physiological data (such as `wblc` and `age`), we implement a `KNNImputer`. This algorithm searches for the 5 nearest patients with mathematically similar multidimensional clinical profiles and fills the missing data with the average of those biological peers, preserving the physiological integrity of the dataset.

**Why KNN Imputation is Reasonable for Age and Wblc:**
Applying the `KNNImputer` rather than relying on blunt global averages is highly reasonable and grounded in clinical logic:

* **Clinical Logic for `wblc`:** Laboratory metrics like white blood cell counts do not vary at random; they are deeply interconnected with a patient's underlying syndromic state. The KNN algorithm exploits these multivariate relationships by finding 5 patients who share the same acute clinical distress profile and leveraging their laboratory values to fill the gap.
* **Biological Logic for `age`:** Age is not just a chronological number; it dictates a patient's baseline physiological reserve and vulnerability to specific diseases. As people age, organ function degrades in predictable patterns, which correlates with shifting baseline metrics in kidney function, vascular elasticity (blood pressure), and comorbidity prevalence (such as dementia or metastatic cancer). If a patient's age is missing, the KNN imputer can successfully infer it by identifying peers who match their specific level of organ degradation and chronic clinical severity. Furthermore, because the proportion of missing data for `age` falls well below the 5% threshold, any minor deviations in the imputed values will not statistically compromise the predictive integrity of the final

The dataset exhibits a negligible proportion of missing values across several other features, specifically: crea, meanbp, hrt, resp, temp, sod, scoma, and aps. To address this sparse missingness, we will also deploy a k-Nearest Neighbors (KNN) imputation strategy. Given the low frequency of these null values, this approach will reliably estimate the missing data points without introducing significant bias or distorting the underlying distribution of the dataset

In [9]:
df = df.drop(columns=['adls'])

income_imputer = SimpleImputer(strategy='most_frequent')
df['income'] = income_imputer.fit_transform(df[['income']]).ravel()

pafi_imputer = SimpleImputer(strategy='median')

df['pafi'] = pafi_imputer.fit_transform(df[['pafi']]).ravel()

c_to_impute = ['wblc','age','crea','meanbp','hrt','resp','temp','sod','scoma','aps']

knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')

df_num_imputed = knn_imputer.fit_transform(df[c_to_impute])

df[c_to_impute] = pd.DataFrame(df_num_imputed, columns=c_to_impute, index=df.index)

print("Missing values in dataframe after KNNInputer:\n", df.isna().sum())


Missing values in dataframe after KNNInputer:
 age          0
sex          0
num_co       0
diabetes     0
dementia     0
meanbp       0
hrt          0
resp         0
temp         0
crea         0
sod          0
adlsc        0
hday         0
ca           0
scoma        0
dzgroup      0
aps          0
wblc         0
pafi         0
income       0
race        42
hospdead     0
dnr         30
dtype: int64


Lets analyze completness.

In [10]:
df.info()
print("Completeness: ", round((1 - df.isna().sum().sum() / df.size) * 100, 2), "%")

<class 'pandas.DataFrame'>
RangeIndex: 9105 entries, 0 to 9104
Data columns (total 23 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       9105 non-null   float64
 1   sex       9105 non-null   str    
 2   num_co    9105 non-null   int64  
 3   diabetes  9105 non-null   int64  
 4   dementia  9105 non-null   int64  
 5   meanbp    9105 non-null   float64
 6   hrt       9105 non-null   float64
 7   resp      9105 non-null   float64
 8   temp      9105 non-null   float64
 9   crea      9105 non-null   float64
 10  sod       9105 non-null   float64
 11  adlsc     9105 non-null   float64
 12  hday      9105 non-null   int64  
 13  ca        9105 non-null   str    
 14  scoma     9105 non-null   float64
 15  dzgroup   9105 non-null   str    
 16  aps       9105 non-null   float64
 17  wblc      9105 non-null   float64
 18  pafi      9105 non-null   float64
 19  income    9105 non-null   str    
 20  race      9063 non-null   str    
 21  ho

As can be seen from the previous df.info() we have still 42 missing data in "race" col , because it just rapresents around  0.5% of the data , we can just delete those rows and this will not impact drastically on the model

In [11]:
df = df.dropna(subset=['race'])
df.to_csv('dataset_clean_tidy_filled.csv', index=False)
df.info()
print("Completeness: ", round((1 - df.isna().sum().sum() / df.size) * 100, 2), "%")

<class 'pandas.DataFrame'>
Index: 9063 entries, 0 to 9104
Data columns (total 23 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       9063 non-null   float64
 1   sex       9063 non-null   str    
 2   num_co    9063 non-null   int64  
 3   diabetes  9063 non-null   int64  
 4   dementia  9063 non-null   int64  
 5   meanbp    9063 non-null   float64
 6   hrt       9063 non-null   float64
 7   resp      9063 non-null   float64
 8   temp      9063 non-null   float64
 9   crea      9063 non-null   float64
 10  sod       9063 non-null   float64
 11  adlsc     9063 non-null   float64
 12  hday      9063 non-null   int64  
 13  ca        9063 non-null   str    
 14  scoma     9063 non-null   float64
 15  dzgroup   9063 non-null   str    
 16  aps       9063 non-null   float64
 17  wblc      9063 non-null   float64
 18  pafi      9063 non-null   float64
 19  income    9063 non-null   str    
 20  race      9063 non-null   str    
 21  hospdea